# Compare the authors' prior with ours trained on compressed AMASS

Two unconditional pose priors with the same architecture and settings (12 flow layers, lr 0.005, batch 2000, seed 10):

| | model | training data | iterations |
|---|---|---|---|
| **(A) authors' prior** | `id_0900` from `experiments/pretrained_models`, as released with the HuProSO3 paper | the full AMASS train split of the authors, most likely the SMPL-X **N** (neutral) release | 20,000 |
| **(B) our prior** | trained with `norm_flow_train_compressed.ipynb` (`OUR_RUN`) | every 10th frame of our local AMASS train split, the SMPL-X **G** (gender-specific) release | 2,000 |

Both get about 2.7 epochs over their training data.

Both are evaluated on the same data: the **full** (not compressed) official test split of our local AMASS, Transitions + SSM (`FULL_ROTATIONS`). The official split is used even if `config/paths.yaml` says `subset`.

Metrics:
- the average log-likelihood of `num_eval_samples` random test poses, repeated a few times (higher is better);
- the Wasserstein distance of generated poses to the test data (lower is better).

(A) shows how the released prior transfers to our data. The difference between (A) and (B) mixes two effects: the subsampling and the different AMASS release.

In [ ]:
import importlib
import json
import os
from pathlib import Path

import numpy as np
import torch
from omegaconf import OmegaConf

from hp.core.evaluator import PriorEvaluator
from hp.data.amass_cfg import ORIGINAL_AMASS_SPLITS
from hp.utils.paths import REPO_ROOT

# ---- parameters ----------------------------------------------------------------------
AUTHORS_PRIOR = REPO_ROOT / "experiments" / "pretrained_models" / "id_0900"   # (A)
OUR_RUN = "~/data/experiments/prior_c10/id_9001_0"                            # (B), output of norm_flow_train_compressed.ipynb
COMPRESSED_RAW = "~/data/amass_compressed"   # compressed AMASS of (B), only read for the frame stride
FULL_ROTATIONS = "~/data/AMASS_rotations"    # rotation files of the FULL data: the test runs on these
DEVICE = "cpu"                               # "cpu", "mps" or "cuda:0"
SEED = 10
REPEATS = 3                                  # log-likelihood evaluations per model
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(p).expanduser()
OUR_RUN = expand(OUR_RUN)

# data of this notebook only (the kernel's environment): the full test split, official AMASS split
os.environ["HUPROSO_AMASS_ROTATIONS"] = str(expand(FULL_ROTATIONS))
os.environ["HUPROSO_AMASS_SPLIT"] = "original"
os.environ.pop("HUPROSO_SUBSET", None)

n_full_test = sum(torch.load(expand(FULL_ROTATIONS) / f"{n}.pt").shape[0] for n in ORIGINAL_AMASS_SPLITS["test"])
print(f"full test data: {expand(FULL_ROTATIONS)} ({', '.join(ORIGINAL_AMASS_SPLITS['test'])}), {n_full_test:,} frames")


def evaluate(model_dir):
    """Log-likelihood and Wasserstein distance of a prior on the full test split."""
    c = OmegaConf.load(Path(model_dir) / "config.yaml")
    c.only_eval, c.standard_mode, c.condition = True, True, False
    c.device = DEVICE
    c.save.path_dir = str(model_dir)
    np.random.seed(SEED), torch.manual_seed(SEED)
    trainer = getattr(importlib.import_module("hp.core.trainer"), c.trainer)(c)
    trainer.load_model(save_path=str(model_dir))
    n_test = len(trainer.eval_dataset)
    assert n_test == n_full_test, f"evaluated on {n_test} test frames, the full test set has {n_full_test}"
    ll = [trainer.eval()["eval_avg_ll"] for _ in range(REPEATS)]
    d = PriorEvaluator.eval(c, trainer, render_human=False, plot_samples=False, save_plot=False, compute_d_wasser=True)
    return dict(model_dir=str(model_dir), iterations=int(c.max_iterations), test_frames=n_test,
                ll=float(np.mean(ll)), ll_std=float(np.std(ll)), wasserstein=float(d["wasserstein_distance_ndim"]))


def show(name, r):
    print(f"{name:45s} {r['test_frames']:,} test frames | log-likelihood {r['ll']:8.2f} +- {r['ll_std']:.2f} | Wasserstein {r['wasserstein']:.3f}")


results = {}

## (A) Authors' prior `id_0900`

The released prior of the paper, trained on the full AMASS data of the authors. It is only loaded, nothing is changed.

In [ ]:
NAME_A = "(A) authors' prior id_0900, full data"
results[NAME_A] = evaluate(AUTHORS_PRIOR)
show(NAME_A, results[NAME_A])

## (B) Our prior trained on the compressed data

Trained with `explore/training/norm_flow_train_compressed.ipynb` on the compressed AMASS (`scripts/preprocess/compress_amass.py`).

In [ ]:
stride = json.load(open(expand(COMPRESSED_RAW) / "compression.json"))["params"]["stride"]
NAME_B = f"(B) our prior, every {stride}-th frame"
results[NAME_B] = evaluate(OUR_RUN)
show(NAME_B, results[NAME_B])

## Summary

Both on the full test split of our local AMASS. Saved as `eval_full_test.json` in the run folder of (B).

In [ ]:
for name, r in results.items():
    show(f"{name}, {r['iterations'] // 1000}k it.", r)
a, b = results[NAME_A], results[NAME_B]
print(f"(B) - (A): log-likelihood {b['ll'] - a['ll']:+.2f}, Wasserstein {b['wasserstein'] - a['wasserstein']:+.3f}")
with open(OUR_RUN / "eval_full_test.json", "w") as f:
    json.dump(results, f, indent=2)